In [1]:
import os
import requests
from datetime import datetime
import pandas as pd
from pathlib import Path
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)

def download_companies_marketcap_csv():
    url = "https://companiesmarketcap.com/largest-companies-by-revenue/?download=csv"
    
    headers = {
        "User-Agent": "Mozilla/5.0 (compatible; DataCollector/1.0)"
    }

    try:
        response = requests.get(url, headers=headers, timeout=30)
        response.raise_for_status()
    except requests.RequestException as e:
        logging.error(f"Erro ao baixar o arquivo: {e}")
        return None

    # Cria o diretório Data/CompaniesMarketCap
    data_dir = Path.cwd() / "Data" / "CompaniesMarketCap"
    data_dir.mkdir(parents=True, exist_ok=True)

    # Nome do arquivo com timestamp
    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    file_name = f"companies_marketcap_{timestamp}.csv"
    file_path = data_dir / file_name

    # Salva o CSV
    with open(file_path, "wb") as file:
        file.write(response.content)

    logging.info(f"Arquivo salvo em: {file_path}")

    # Tenta ler o CSV
    try:
        df = pd.read_csv(file_path, encoding="utf-8")
    except UnicodeDecodeError:
        df = pd.read_csv(file_path, encoding="latin1")

    logging.info("Primeiras linhas do arquivo:")
    logging.info(f"\n{df.head()}")

    return df


if __name__ == "__main__":
    download_companies_marketcap_csv()


2026-01-22 17:15:22,250 - INFO - Arquivo salvo em: c:\Users\felip\Documents\GitHub\TCC\NOMS-2026\pipeline\src\Data\CompaniesMarketCap\companies_marketcap_20260122_171522.csv
2026-01-22 17:15:22,325 - INFO - Primeiras linhas do arquivo:
2026-01-22 17:15:22,346 - INFO - 
   Rank          Name     Symbol   revenue_ttm  price (USD)        country
0     1       Walmart        WMT  703061000000   117.950000  United States
1     2        Amazon       AMZN  691330000000   234.630000  United States
2     3  Saudi Aramco    2222.SR  461568118758     6.730850   Saudi Arabia
3     4       Sinopec  600028.SS  444749587530     0.890629          China
4     5  UnitedHealth        UNH  435159000000   354.560000  United States


In [ ]:
#selecionar arquivo mais recente
def get_most_recent_file():
    data_dir = Path.cwd() / "Data" / "CompaniesMarketCap"
    csv_files = list(data_dir.glob("companies_marketcap_*.csv"))
    
    if not csv_files:
        logging.warning("Nenhum arquivo encontrado no diretório.")
        return None

    most_recent_file = max(csv_files, key=os.path.getctime)
    logging.info(f"Arquivo mais recente: {most_recent_file}")
    return most_recent_file

